# MCP-RSS Server — Google Colab Test Notebook

This notebook tests the **mcp-rss** Python MCP server using the `rss_search_mcp.py` client script.

## Architecture Overview

```
rss_search_mcp.py  (MCP client)
        │
        │  stdio transport
        ▼
  src/server.py  (MCP server — FastMCP)
        │
        ├── fetch_rss_feed        – single feed fetch
        ├── fetch_multiple_feeds  – parallel batch fetch
        ├── search_feed_items     – full-text search across feeds
        ├── monitor_feed_updates  – new items since timestamp
        ├── extract_feed_content  – formatted content extraction
        └── get_feed_headlines    – compact headline list
```

### What `rss_search_mcp.py` does
1. Spawns the MCP server as a subprocess over stdio.
2. Calls **`fetch_multiple_feeds`** on 3 aggregator feeds (Google News, Bing News, Yahoo News) with the query baked into their URLs.
3. Calls **`search_feed_items`** on 30 general feeds (Reuters, BBC, AP, NPR, Al Jazeera, etc.) to filter server-side.
4. Merges, deduplicates, ranks, and displays up to 15 articles.

> **Colab note:** Jupyter runs an event loop in the kernel. `asyncio.run()` will raise
> `RuntimeError: This event loop is already running` unless `nest_asyncio` is applied first.
> Section 2 installs and applies it automatically.

## 1 — Clone the Repository

In [1]:
import os

REPO_URL = "https://github.com/dshipley71/mcp-servers.git"
REPO_DIR = "/content/mcp-servers"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already cloned at {REPO_DIR}")
    !git -C {REPO_DIR} pull

Cloning into '/content/mcp-servers'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 97 (delta 19), reused 13 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 109.41 KiB | 2.67 MiB/s, done.
Resolving deltas: 100% (19/19), done.


## 2 — Install Dependencies

`nest_asyncio` is required because Colab's Jupyter kernel already runs an asyncio event loop.
Applying it allows `asyncio.run()` to work correctly inside notebook cells.

In [2]:
!pip install -q \
    "mcp[cli]>=1.0.0" \
    "httpx>=0.27.0" \
    "feedparser>=6.0.11" \
    "beautifulsoup4>=4.12.0" \
    "bleach>=6.1.0" \
    "python-dotenv>=1.0.0" \
    "pydantic>=2.7.0" \
    "pydantic-settings>=2.3.0" \
    "rich>=13.0.0" \
    "python-dateutil" \
    "nest_asyncio"

# ── Patch the running event loop so asyncio.run() works inside Jupyter ──
import nest_asyncio
nest_asyncio.apply()

print("All dependencies installed and event-loop patch applied.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.8 MB/s eta 0:00:00
All dependencies installed and event-loop patch applied.


## 3 — Verify Repository Structure

In [3]:
import os
from pathlib import Path

MCP_RSS_DIR   = Path(REPO_DIR) / "mcp-rss"
SERVER_SCRIPT = MCP_RSS_DIR / "src" / "server.py"
SEARCH_SCRIPT = MCP_RSS_DIR / "rss_search_mcp.py"

checks = {
    "mcp-rss directory":             MCP_RSS_DIR.is_dir(),
    "src/server.py (MCP server)":     SERVER_SCRIPT.is_file(),
    "rss_search_mcp.py (client)":     SEARCH_SCRIPT.is_file(),
    "src/config.py":                  (MCP_RSS_DIR / "src" / "config.py").is_file(),
    "src/services/rss_reader.py":     (MCP_RSS_DIR / "src" / "services" / "rss_reader.py").is_file(),
    "src/services/cache.py":          (MCP_RSS_DIR / "src" / "services" / "cache.py").is_file(),
}

all_ok = True
for name, ok in checks.items():
    status = "OK" if ok else "MISSING"
    print(f"[{status:7s}]  {name}")
    if not ok:
        all_ok = False

if all_ok:
    print("\nAll required files found — ready to run.")
else:
    print("\nWARNING: Some files are missing. Check the clone step above.")

[OK     ]  mcp-rss directory
[OK     ]  src/server.py (MCP server)
[OK     ]  rss_search_mcp.py (client)
[OK     ]  src/config.py
[OK     ]  src/services/rss_reader.py
[OK     ]  src/services/cache.py

All required files found — ready to run.


## 4 — Configure the Environment

`rss_search_mcp.py` resolves the MCP server directory from `MCP_RSS_SERVER_DIR`.
We point it at `mcp-rss/` which contains `src/server.py`.
We also add `mcp-rss/` to `sys.path` so we can import `src.*` directly.

In [4]:
import os, sys

# Tell rss_search_mcp.py where the MCP server lives.
os.environ["MCP_RSS_SERVER_DIR"] = str(MCP_RSS_DIR)

# Add mcp-rss/ to sys.path so direct imports (src.*) work.
mcp_rss_str = str(MCP_RSS_DIR)
if mcp_rss_str not in sys.path:
    sys.path.insert(0, mcp_rss_str)

print(f"MCP_RSS_SERVER_DIR = {os.environ['MCP_RSS_SERVER_DIR']}")
print(f"Server entry-point : {SERVER_SCRIPT}")
print(f"Client script      : {SEARCH_SCRIPT}")
print(f"sys.path[0]        : {sys.path[0]}")

MCP_RSS_SERVER_DIR = /content/mcp-servers/mcp-rss
Server entry-point : /content/mcp-servers/mcp-rss/src/server.py
Client script      : /content/mcp-servers/mcp-rss/rss_search_mcp.py
sys.path[0]        : /content/mcp-servers/mcp-rss


## 5 — Smoke-Test the MCP Server (Direct)

Confirm the server starts without errors by exercising the RSS reader directly.
This also initialises the module-level singletons (`feed_cache`, `rss_reader`) that
later sections reuse.

In [5]:
import asyncio
from src.services.rss_reader import rss_reader
from src.services.cache import feed_cache

# Start the background cache-cleanup task.
# feed_cache.start() schedules an asyncio.Task, so it must run inside
# an active event loop — wrapping it in a coroutine achieves this.
async def _start_cache():
    feed_cache.start()

asyncio.run(_start_cache())

TEST_FEED = "https://feeds.bbci.co.uk/news/world/rss.xml"

async def smoke_test():
    print(f"Fetching test feed: {TEST_FEED}")
    result = await rss_reader.fetch_feed(TEST_FEED)
    print(f"\nFeed title   : {result.info.title}")
    print(f"Items fetched: {len(result.items)}")
    if result.items:
        first = result.items[0]
        print(f"\nFirst item   : {first.title}")
        print(f"URL          : {first.url}")
    return result

feed_result = asyncio.run(smoke_test())
print("\nMCP RSS reader is working correctly.")

Fetching test feed: https://feeds.bbci.co.uk/news/world/rss.xml

Feed title   : BBC News
Items fetched: 24

First item   : Who is Mojtaba Khamenei, Iran's new supreme leader?
URL          : https://www.bbc.com/news/articles/c78xxg05w0zo?at_medium=RSS&at_campaign=rss

MCP RSS reader is working correctly.


## 6 — Exercise Individual MCP Server Tools

The cells below call each of the six MCP tools directly (without the stdio client)
so you can inspect their output in isolation.

### Tool 1 — `fetch_rss_feed`
Fetch and parse a single RSS/Atom feed.

In [6]:
import json
from src.server import fetch_rss_feed

async def test_fetch_rss_feed():
    raw = await fetch_rss_feed("https://feeds.bbci.co.uk/news/world/rss.xml")
    data = json.loads(raw)
    info  = data.get("info", {})
    items = data.get("items", [])
    print(f"Feed : {info.get('title')}")
    print(f"Items: {len(items)}")
    print("\nFirst 3 headlines:")
    for art in items[:3]:
        print(f"  * {art.get('title')}")
    return data

feed_data = asyncio.run(test_fetch_rss_feed())

[2026-03-08T22:23:37.637923+00:00] [INFO] Fetching RSS feed: https://feeds.bbci.co.uk/news/world/rss.xml
INFO:mcp-rss:Fetching RSS feed: https://feeds.bbci.co.uk/news/world/rss.xml
[2026-03-08T22:23:37.782759+00:00] [INFO] Successfully fetched feed: https://feeds.bbci.co.uk/news/world/rss.xml, 24 items
INFO:mcp-rss:Successfully fetched feed: https://feeds.bbci.co.uk/news/world/rss.xml, 24 items


Feed : BBC News
Items: 24

First 3 headlines:
  * Who is Mojtaba Khamenei, Iran's new supreme leader?
  * War fuels debate in Cyprus over UK military bases
  * Explosion at US embassy in Oslo may have been terrorism, Norway police say


### Tool 2 — `fetch_multiple_feeds`
Batch-fetch several feeds in parallel.

In [7]:
from src.server import fetch_multiple_feeds

SAMPLE_FEEDS = [
    "https://feeds.bbci.co.uk/news/world/rss.xml",
    "https://feeds.npr.org/1001/rss.xml",
    "https://rss.dw.com/rdf/rss-en-all",
]

async def test_fetch_multiple_feeds():
    raw  = await fetch_multiple_feeds(SAMPLE_FEEDS, parallel="true")
    data = json.loads(raw)
    print(f"Total   : {data['total']}")
    print(f"Success : {data['successful']}")
    print(f"Failed  : {data['failed']}")
    print()
    for r in data["results"]:
        ok    = r["success"]
        items = len(r["data"]["items"]) if r.get("data") else 0
        status = "OK" if ok else "FAIL"
        print(f"[{status}]  {r['url']}  ({items} items)")
    return data

multi_data = asyncio.run(test_fetch_multiple_feeds())

[2026-03-08T22:23:40.069425+00:00] [INFO] Fetching 3 feeds (parallel: true)
INFO:mcp-rss:Fetching 3 feeds (parallel: true)
[2026-03-08T22:23:40.599673+00:00] [INFO] Fetched 3/3 feeds successfully
INFO:mcp-rss:Fetched 3/3 feeds successfully


Total   : 3
Success : 3
Failed  : 0

[OK]  https://feeds.bbci.co.uk/news/world/rss.xml  (24 items)
[OK]  https://feeds.npr.org/1001/rss.xml  (10 items)
[OK]  https://rss.dw.com/rdf/rss-en-all  (100 items)


### Tool 3 — `search_feed_items`
Full-text search across multiple feeds.

In [8]:
from src.server import search_feed_items

SEARCH_QUERY = "climate"  # change this to any topic

async def test_search_feed_items():
    raw  = await search_feed_items(
        feeds=SAMPLE_FEEDS,
        query=SEARCH_QUERY,
        search_in="all",
    )
    data = json.loads(raw)
    print(f"Query          : {data['query']!r}")
    print(f"Feeds searched : {data['feedsSearched']}")
    print(f"Total matches  : {data['totalMatches']}")
    print()
    for hit in data["results"][:5]:
        item = hit["item"]
        print(f"  [{', '.join(hit['matches'])}]  {item.get('title')}")
        print(f"         Source : {hit['feedTitle']}")
        print()
    return data

search_data = asyncio.run(test_search_feed_items())

[2026-03-08T22:23:45.793257+00:00] [INFO] Searching 3 feeds for: "climate"
INFO:mcp-rss:Searching 3 feeds for: "climate"
[2026-03-08T22:23:45.795996+00:00] [INFO] Found 2 matching items
INFO:mcp-rss:Found 2 matching items


Query          : 'climate'
Feeds searched : 3
Total matches  : 2

  [description]  Why it's worth getting a heat pump if you live in a cold country, too
         Source : Deutsche Welle

  [description]  Is sustainable travel possible in times of mass tourism?
         Source : Deutsche Welle



### Tool 4 — `monitor_feed_updates`
Return only items published since a given timestamp.

In [9]:
import time
from src.server import monitor_feed_updates

# Look for items from the past 24 hours
since_ms = int((time.time() - 86400) * 1000)

async def test_monitor_feed_updates():
    raw  = await monitor_feed_updates(
        url="https://feeds.bbci.co.uk/news/world/rss.xml",
        since=since_ms,
    )
    data = json.loads(raw)
    print(f"Feed           : {data['feedTitle']}")
    print(f"Since          : {data['sinceISO']}")
    print(f"New items      : {data['newItemsCount']}")
    print(f"Total items    : {data['totalItemsCount']}")
    print()
    for item in data["newItems"][:5]:
        print(f"  * {item.get('title')}")
    return data

monitor_data = asyncio.run(test_monitor_feed_updates())

[2026-03-08T22:23:51.932497+00:00] [INFO] Monitoring updates for: https://feeds.bbci.co.uk/news/world/rss.xml since 1772922231931
INFO:mcp-rss:Monitoring updates for: https://feeds.bbci.co.uk/news/world/rss.xml since 1772922231931
[2026-03-08T22:23:52.067004+00:00] [INFO] Found 14 new items since 2026-03-07T22:23:51.931000+00:00
INFO:mcp-rss:Found 14 new items since 2026-03-07T22:23:51.931000+00:00


Feed           : BBC News
Since          : 2026-03-07T22:23:51.931000+00:00
New items      : 14
Total items    : 24

  * Who is Mojtaba Khamenei, Iran's new supreme leader?
  * War fuels debate in Cyprus over UK military bases
  * Explosion at US embassy in Oslo may have been terrorism, Norway police say
  * Boy, 12, among six dead as tornadoes hit Michigan and Oklahoma
  * Swiss reject right-wing plan to cut licence fee for public broadcaster


### Tool 5 — `extract_feed_content`
Extract and format feed content as markdown, text, HTML, or JSON.

In [10]:
from src.server import extract_feed_content
from IPython.display import Markdown, display

async def test_extract_feed_content():
    raw  = await extract_feed_content(
        url="https://feeds.bbci.co.uk/news/world/rss.xml",
        format="markdown",
        include_metadata="true",
    )
    data = json.loads(raw)
    print(f"Feed       : {data['feedTitle']}")
    print(f"Items      : {data['itemCount']}")
    print(f"Format     : {data['format']}")
    print("\n--- First item rendered as Markdown ---\n")
    first_md = data["content"][0] if data["content"] else "(no content)"
    display(Markdown(first_md))
    return data

content_data = asyncio.run(test_extract_feed_content())

[2026-03-08T22:23:56.699165+00:00] [INFO] Extracting content from: https://feeds.bbci.co.uk/news/world/rss.xml (format: markdown)
INFO:mcp-rss:Extracting content from: https://feeds.bbci.co.uk/news/world/rss.xml (format: markdown)
[2026-03-08T22:23:56.704566+00:00] [INFO] Extracted content from 24 items
INFO:mcp-rss:Extracted content from 24 items


Feed       : BBC News
Items      : 24
Format     : markdown

--- First item rendered as Markdown ---



# Who is Mojtaba Khamenei, Iran's new supreme leader?

Title: Who is Mojtaba Khamenei, Iran's new supreme leader?
Published: 2026-03-08T21:51:48+00:00
URL: https://www.bbc.com/news/articles/c78xxg05w0zo?at_medium=RSS&at_campaign=rss

Many expect the 56-year-old, who has largely kept a low profile, to continue his father's hardline policies.


[Read more](https://www.bbc.com/news/articles/c78xxg05w0zo?at_medium=RSS&at_campaign=rss)

### Tool 6 — `get_feed_headlines`
Return a compact headline list (title, summary, URL).

In [11]:
from src.server import get_feed_headlines

async def test_get_feed_headlines():
    raw  = await get_feed_headlines(
        url="https://feeds.npr.org/1001/rss.xml",
        format="json",
    )
    data = json.loads(raw)
    print(f"Feed       : {data['feedTitle']}")
    print(f"Headlines  : {data['itemCount']}")
    print()
    for h in data["headlines"][:5]:
        print(f"  Title   : {h.get('title')}")
        print(f"  URL     : {h.get('url')}")
        print()
    return data

headlines_data = asyncio.run(test_get_feed_headlines())

[2026-03-08T22:24:07.739056+00:00] [INFO] Getting headlines from: https://feeds.npr.org/1001/rss.xml
INFO:mcp-rss:Getting headlines from: https://feeds.npr.org/1001/rss.xml
[2026-03-08T22:24:07.744295+00:00] [INFO] Got 10 headlines from https://feeds.npr.org/1001/rss.xml
INFO:mcp-rss:Got 10 headlines from https://feeds.npr.org/1001/rss.xml


Feed       : NPR Topics: News
Headlines  : 10

  Title   : OpenAI robotics leader resigns over concerns about Pentagon AI deal
  URL     : https://www.npr.org/2026/03/08/nx-s1-5741779/openai-resigns-ai-pentagon-guardrails-military

  Title   : Trump says he won't sign bills until Congress overhauls voting
  URL     : https://www.npr.org/2026/03/08/g-s1-112917/trump-says-he-wont-sign-bills-until-congress-overhauls-voting

  Title   : Photos: Scenes from Jesse Jackson's homegoing services
  URL     : https://www.npr.org/sections/the-picture-show/2026/03/08/g-s1-112915/photos-scenes-from-jesse-jacksons-homegoing-services

  Title   : Five key takeaways from an annual briefing by China's foreign minister
  URL     : https://www.npr.org/2026/03/08/nx-s1-5741742/five-key-takeaways-from-an-annual-briefing-by-chinas-foreign-minister

  Title   : Police investigate an explosion outside the U.S. Embassy in Oslo
  URL     : https://www.npr.org/2026/03/08/nx-s1-5741670/explosion-outside-us-embassy

## 7 — Full End-to-End Search via `rss_search_mcp.py`

This section runs `rss_search_mcp.py` as designed — it spawns the MCP server as a
**subprocess** over stdio and exercises both MCP tools concurrently across all 33 feeds.

Rich console output goes to **stderr**; JSON results go to **stdout**.  
Both streams are captured and printed below.  
`NO_COLOR=1` disables ANSI escape codes so output renders cleanly in the notebook.

### 7a — Configure Your Search Query

In [12]:
# ── Customise these ──────────────────────────────────────────────────────────
QUERY        = "artificial intelligence"  # topic to search for
SAVE_RESULTS = True                       # write results to a JSON file?
RESULTS_FILE = "/content/rss_results.json"
# ─────────────────────────────────────────────────────────────────────────────

print(f"Query        : {QUERY!r}")
print(f"Save to file : {SAVE_RESULTS} -> {RESULTS_FILE if SAVE_RESULTS else 'n/a'}")

Query        : 'artificial intelligence'
Save to file : True -> /content/rss_results.json


### 7b — Run the Full MCP Search

In [13]:
import re, subprocess, sys, os

def strip_ansi(text: str) -> str:
    """Remove ANSI escape sequences for clean notebook output."""
    return re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", text)

cmd = [sys.executable, str(SEARCH_SCRIPT), QUERY]
if SAVE_RESULTS:
    cmd += ["--save", RESULTS_FILE]

env = os.environ.copy()
env["MCP_RSS_SERVER_DIR"] = str(MCP_RSS_DIR)
env["NO_COLOR"] = "1"   # ask Rich to suppress colour codes
env["TERM"]     = "dumb" # belt-and-suspenders terminal colour disable

print("Running:", " ".join(cmd))
print("-" * 60)

proc = subprocess.run(
    cmd,
    env=env,
    stdout=subprocess.PIPE,   # capture JSON output
    stderr=subprocess.PIPE,   # capture Rich progress/table output
    text=True,
)

# Print stderr (Rich tables, progress) first, then stdout (JSON)
if proc.stderr:
    print(strip_ansi(proc.stderr))
if proc.stdout:
    print(strip_ansi(proc.stdout))

print("-" * 60)
print(f"Exit code: {proc.returncode}")
if proc.returncode != 0:
    print("ERROR: script exited with non-zero code — check output above.")

Running: /usr/bin/python3 /content/mcp-servers/mcp-rss/rss_search_mcp.py artificial intelligence --save /content/rss_results.json
------------------------------------------------------------
───────────────────── RSS Topic Search  ·  mcp-rss-python ──────────────────────
╭──────────────────────────────────────────────────────────────────────────────╮
│ Query   : artificial intelligence                                            │
│ Server  : /content/mcp-servers/mcp-rss                                       │
│ Feeds   : 34  (3 search + 31 general)                                        │
│ Tools   : fetch_multiple_feeds  +  search_feed_items                         │
│ Limit   : 15 articles                                                        │
╰──────────────────────────────────────────────────────────────────────────────╯
[2026-03-08T22:24:35.808685+00:00] [INFO] MCP-RSS server starting...
[03/08/26 22:24:35] INFO     MCP-RSS server starting...            server.py:547
           

### 7c — Load and Display Saved Results

In [14]:
import json
from pathlib import Path

if SAVE_RESULTS and Path(RESULTS_FILE).exists():
    with open(RESULTS_FILE) as f:
        results = json.load(f)

    print(f"Query              : {results.get('query')}")
    print(f"Generated at       : {results.get('generated_at')}")
    print(f"Total feeds        : {results.get('total_feeds')}")
    print(f"Search feeds OK    : {results.get('search_feeds_ok')}")
    print(f"Search feeds failed: {results.get('search_feeds_failed')}")
    print(f"General feeds      : {results.get('general_feeds_searched')}")
    print(f"Raw matches        : {results.get('raw_matches')}")
    print(f"After dedup        : {results.get('after_dedup')}")
    print(f"Returned articles  : {results.get('total_results')}")
    print()

    for i, art in enumerate(results.get("articles", []), 1):
        pub = (art.get("published_iso") or "---")[:10]
        print(f"{i:>2}. [{pub}] {art.get('title')}")
        print(f"     Source   : {art.get('source')}  ({art.get('source_category')} / {art.get('source_region')})")
        print(f"     URL      : {art.get('url')}")
        summary = (art.get("summary") or "")[:200]
        if summary:
            suffix = "..." if len(art.get("summary", "")) > 200 else ""
            print(f"     Summary  : {summary}{suffix}")
        print()
else:
    print("No saved results file found. Set SAVE_RESULTS = True and re-run section 7b.")

Query              : artificial intelligence
Generated at       : 2026-03-08T22:24:56.677735+00:00
Total feeds        : 34
Search feeds OK    : 2
Search feeds failed: 1
General feeds      : 31
Raw matches        : 121
After dedup        : 109
Returned articles  : 15

 1. [2026-03-08] I’m begging photographers not to use AI to write their Instagram captions. Here’s why it’s such a bad idea - Digital Camera World
     Source   : Google News  (Aggregator / Global)
     URL      : https://news.google.com/rss/articles/CBMi-gFBVV95cUxPUGplb3U0cDNEcWdVRXdhNVRhSDVrU1FZaFNSclAtMTN4b2c4aldYN2FYYWZrNnl3RXpPRVR5RnVUbHNmWkd5bE9QNk5YRXBVMExBTzNqeDcxZWUyeFMwYUJqeWxWUldKemE5UDFpQWNWb1JrLUluZFRvOHdOdzhHWkVJc25HdEUyOUowS1dzX09mSVJwMnpVaW83aktYWG9DRm9uQkd4ZjZLQVI1bUhpUjFYcXp1UGpJTFpqWVNKUlBVUVQ3eXIyV0dmaGRabl9tcmZESXpKSlhKMUdFUGtVQ0tWbTZwQWYzSWt2WkRLeU5IUFRZT083Q013?oc=5
     Summary  : I’m begging photographers not to use AI to write their Instagram captions. Here’s why it’s such a bad idea Digital Came

## 8 — Programmatic MCP Search (Inline, No Subprocess)

This section imports `_run_search` and helpers directly from `rss_search_mcp.py`
so you can iterate programmatically without touching the original script.

Internally `_run_search` still spawns the MCP server over stdio — it just does so
from inside the current Python process via the MCP `stdio_client`.

In [15]:
import importlib.util

# Load rss_search_mcp.py as a module (MCP_RSS_SERVER_DIR env var is already set).
spec    = importlib.util.spec_from_file_location("rss_search_mcp", str(SEARCH_SCRIPT))
rss_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rss_mod)

MY_QUERY = "climate change"  # <- change this

raw_articles, stats = asyncio.run(rss_mod._run_search(MY_QUERY))

# Sort and deduplicate (mirrors the logic in main())
raw_articles.sort(key=lambda a: (
    0 if "search_feed" in (a.get("matched_fields") or []) else 1,
    -(a.get("published_epoch") or 0),
))
deduped = rss_mod._deduplicate(raw_articles)
final   = deduped[:rss_mod.MAX_ARTICLES]

print(f"Query          : {MY_QUERY!r}")
print(f"Raw matches    : {stats['raw_matches']}")
print(f"After dedup    : {len(deduped)}")
print(f"Returned       : {len(final)}")
print()

for i, art in enumerate(final, 1):
    pub = (art.get("published_iso") or "---")[:10]
    print(f"{i:>2}. [{pub}] {art.get('title')}")
    print(f"     {art.get('source')} -- {art.get('source_category')}")
    print()

UnsupportedOperation: fileno

## 9 — Results as a Pandas DataFrame

In [ ]:
import pandas as pd

# Use whichever results are available: inline search (section 8) or saved file (section 7c).
if "final" in dir() and final:
    articles_for_df = final
    df_label = f"Inline search: {MY_QUERY!r}"
elif "results" in dir() and results.get("articles"):
    articles_for_df = results["articles"]
    df_label = f"Subprocess search: {results.get('query')!r}"
else:
    articles_for_df = []
    df_label = "No results"

if articles_for_df:
    df = pd.DataFrame(articles_for_df)[
        ["title", "source", "source_category", "source_region",
         "published_iso", "matched_fields", "url"]
    ].copy()

    df["published"]      = df["published_iso"].str[:10]
    df["matched_fields"] = df["matched_fields"].apply(lambda v: ", ".join(v or []))

    display_cols = ["title", "source", "source_category", "source_region",
                    "published", "matched_fields"]
    print(f"Showing: {df_label}  ({len(df)} rows)")
    display(df[display_cols])
else:
    print("No articles to display. Run section 7b or 8 first.")

## 10 — Inspect the Feed Cache

In [ ]:
from src.services.cache import feed_cache

stats = feed_cache.get_stats()
# get_stats() returns: {"size": int, "urls": list[str]}
print(f"Cache entries : {stats.get('size', 0)}")
print()

cached_urls = stats.get("urls", [])
print(f"Cached feeds ({len(cached_urls)}):")
for url in cached_urls[:10]:
    cached = feed_cache.get(url)
    items  = len(cached.items) if cached else 0
    title  = cached.info.title if cached else "(unavailable)"
    print(f"  {title[:40]:40s}  {items:3d} items  {url}")

if len(cached_urls) > 10:
    print(f"  ... and {len(cached_urls) - 10} more")

## 11 — Multi-Query Batch Search

Run several queries sequentially and collect all results.
`asyncio.run()` inside a for loop works because `nest_asyncio` was applied in section 2.

In [ ]:
QUERIES = [
    "artificial intelligence",
    "election 2025",
    "climate summit",
]

all_results: dict[str, list] = {}

for q in QUERIES:
    print(f"Searching for {q!r} ...", end=" ", flush=True)
    arts, st = asyncio.run(rss_mod._run_search(q))
    arts.sort(key=lambda a: (
        0 if "search_feed" in (a.get("matched_fields") or []) else 1,
        -(a.get("published_epoch") or 0),
    ))
    final_q = rss_mod._deduplicate(arts)[:rss_mod.MAX_ARTICLES]
    all_results[q] = final_q
    print(f"{len(final_q)} articles")

print()
print("Summary:")
for q, arts in all_results.items():
    print(f"  {q!r:35s}  {len(arts):2d} articles")

## 12 — Export All Results to JSON

In [ ]:
from datetime import datetime, timezone

EXPORT_FILE = "/content/mcp_rss_batch_results.json"

export = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "queries": [
        {
            "query":    q,
            "count":    len(arts),
            "articles": arts,
        }
        for q, arts in all_results.items()
    ],
}

with open(EXPORT_FILE, "w") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)

total = sum(len(a) for a in all_results.values())
print(f"Exported {total} articles across {len(all_results)} queries to {EXPORT_FILE}")

# Download the file in Colab
try:
    from google.colab import files
    files.download(EXPORT_FILE)
except ImportError:
    print("(Not running in Colab — skipping download prompt)")

## 13 — Cleanup

In [ ]:
# Cancel the background cleanup task and clear all cached feeds.
feed_cache.destroy()
print("Feed cache shut down cleanly.")